# IV. Huấn luyện và so sánh mô hình

Notebook so sánh hai nhóm mô hình trên cùng 104 trận WC 2026:

1. **Mô hình tỉ số** — dự đoán số bàn của từng đội rồi suy ra thắng/hoà/thua: Poisson Regression, Ridge Regression, Random Forest, Extra Trees.
2. **Mô hình 1X2 trực tiếp** — dự đoán thẳng xác suất thắng/hoà/thua: Multinomial Logit.

Huấn luyện trên các trận từ 14/06/2018 đến 10/06/2026. Tập kiểm tra là 104 trận WC 2026 (11/06 – 19/07/2026).


## 1. Nạp dữ liệu

Chạy `wc.ipynb` trước. Hai file được đọc từ `loc/` đều do `wc.ipynb` sinh ra, nên Elo của tập train và tập test cùng một thang đo.

Bộ feature gồm 7 cột: Elo hai đội trước trận, cờ sân trung lập, và 4 chỉ số bàn ghi/thủng trung bình trong tối đa 5 trận trước đó.


In [97]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'loc').is_dir():
    ROOT = ROOT.parent

def read_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as file:
        return list(csv.DictReader(file))

def parse_bool(value):
    return str(value).strip().casefold() in {'true', '1', 'yes'}

history = read_csv(ROOT / 'loc' / 'matches_truoc_wc_2026_elo_features.csv')
world_cup = read_csv(ROOT / 'loc' / 'wc_2026_da_loc.csv')

FEATURE_NAMES = [
    'home_elo_pre', 'away_elo_pre', 'neutral',
    'home_form_goals_scored_last5', 'home_form_goals_conceded_last5',
    'away_form_goals_scored_last5', 'away_form_goals_conceded_last5',
]

def make_data(rows):
    X = np.asarray([
        [
            float(row['home_elo_pre']),
            float(row['away_elo_pre']),
            float(parse_bool(row['neutral'])),
            float(row['home_form_goals_scored_last5']),
            float(row['home_form_goals_conceded_last5']),
            float(row['away_form_goals_scored_last5']),
            float(row['away_form_goals_conceded_last5']),
        ]
        for row in rows
    ], dtype=float)
    y_home = np.asarray([float(row['home_score']) for row in rows])
    y_away = np.asarray([float(row['away_score']) for row in rows])
    return X, y_home, y_away

train_data = make_data(history)
wc2026_data = make_data(world_cup)

X_train, y_home_train, y_away_train = train_data
X_test, y_home_test, y_away_test = wc2026_data

print(f'Tập huấn luyện: {len(history):,} trận ({history[0]["date"]} đến {history[-1]["date"]})')
print(f'Tập kiểm tra WC2026: {len(world_cup):,} trận ({world_cup[0]["date"]} đến {world_cup[-1]["date"]})')

print(f'Elo trung bình - train: {X_train[:, 0].mean():.0f} (nhà) / {X_train[:, 1].mean():.0f} (khách)')
print(f'Elo trung bình - test : {X_test[:, 0].mean():.0f} (nhà) / {X_test[:, 1].mean():.0f} (khách)')
print('  Test cao hơn là bình thường: 48 đội dự WC là nhóm mạnh nhất,')
print('  còn tập train gồm cả các đội yếu.')

# Hai tập phải dùng CÙNG thang Elo: so Elo của từng đội ở cuối train với Elo
# đầu tiên của đội đó trong giải. Nếu lệch vài trăm điểm thì đang lấy Elo từ
# hai nguồn khác nhau.
last_train_elo = {}
for row in history:
    last_train_elo[row['home_team']] = float(row['home_elo_pre'])
    last_train_elo[row['away_team']] = float(row['away_elo_pre'])

first_test_elo = {}
for row in world_cup:
    first_test_elo.setdefault(row['home_team'], float(row['home_elo_pre']))
    first_test_elo.setdefault(row['away_team'], float(row['away_elo_pre']))

gaps = [abs(first_test_elo[team] - last_train_elo[team])
        for team in first_test_elo if team in last_train_elo]
print(f'Chênh lệch Elo của cùng một đội giữa cuối train và đầu test: '
      f'trung vị {np.median(gaps):.0f}, lớn nhất {max(gaps):.0f} điểm')

Tập huấn luyện: 7,768 trận (2018-06-14 đến 2026-06-10)
Tập kiểm tra WC2026: 104 trận (2026-06-11 đến 2026-07-19)
Elo trung bình - train: 1598 (nhà) / 1568 (khách)
Elo trung bình - test : 1949 (nhà) / 1896 (khách)
  Test cao hơn là bình thường: 48 đội dự WC là nhóm mạnh nhất,
  còn tập train gồm cả các đội yếu.
Chênh lệch Elo của cùng một đội giữa cuối train và đầu test: trung vị 5, lớn nhất 31 điểm


### Hình dung dữ liệu huấn luyện

Biểu đồ bên trái cho thấy phân bố số bàn trong tập huấn luyện; bên phải cho thấy kết quả trận thay đổi theo chênh lệch Elo ở các nhóm có số trận tương đương. Đây là quan hệ mô tả dữ liệu, không khẳng định Elo là nguyên nhân duy nhất.

![Phân bố bàn thắng và kết quả theo chênh lệch Elo](../figures/train_01_goals_elo.png)

## 2. Bảy feature đầu vào

1. Elo đội nhà trước trận
2. Elo đội khách trước trận
3. Sân trung lập
4. Bàn đội nhà ghi trung bình trong tối đa 5 trận trước đó
5. Bàn đội nhà thủng lưới trung bình trong tối đa 5 trận trước đó
6. Bàn đội khách ghi trung bình trong tối đa 5 trận trước đó
7. Bàn đội khách thủng lưới trung bình trong tối đa 5 trận trước đó

Cả 7 feature được chuẩn hoá bằng `StandardScaler` trước khi đưa vào mô hình, nên Elo (thang ~1500–2200) và số bàn trung bình (thang ~1) không làm lệch hệ số.


In [98]:
# alpha là hệ số regularization của Poisson Regression. alpha càng lớn thì
# hệ số của các feature bị kéo về 0 càng mạnh, tức là mô hình càng "phẳng".
# alpha = 1.0 (giá trị mặc định hay dùng) quá lớn với 7 feature này: hệ số Elo
# bị co từ 0.44 xuống 0.17, nên mô hình không còn phân biệt được đội mạnh với
# đội yếu. Đo trên 469 trận của 8 giải, hạ alpha xuống 0.01 giúp RPS từ 0.1984
# xuống 0.1881 và độ chính xác thắng/hoà/thua từ 52.0% lên 56.5%.
POISSON_ALPHA = 0.01

models = {
    'Poisson Regression': lambda: make_pipeline(
        StandardScaler(), PoissonRegressor(alpha=POISSON_ALPHA, max_iter=3000)
    ),
    'Ridge Regression': lambda: make_pipeline(
        StandardScaler(), Ridge(alpha=10.0)
    ),
    'Random Forest': lambda: RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    'Extra Trees': lambda: ExtraTreesRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
}
print(f'{len(models)} mô hình sẽ được so sánh:', ', '.join(models))

4 mô hình sẽ được so sánh: Poisson Regression, Ridge Regression, Random Forest, Extra Trees


## 3. Lý do chọn bốn mô hình

- **Poisson Regression**: mô hình nền chuyên cho dữ liệu đếm như số bàn thắng.
- **Ridge Regression**: mô hình tuyến tính có regularization L2, làm mốc so sánh đơn giản.
- **Random Forest**: kết hợp nhiều cây, học quan hệ phi tuyến và ít nhạy với một cây đơn lẻ.
- **Extra Trees**: tổ hợp cây ngẫu nhiên với cách chia ngưỡng khác Random Forest.

Mỗi mô hình dự đoán số bàn kỳ vọng λ cho hai đội. Từ hai giá trị λ, ta dựng ma trận xác suất tỉ số rồi chọn ô có xác suất lớn nhất; cộng xác suất theo từng nhóm để chọn kết quả thắng/hoà/thua.

### Vì sao alpha của Poisson phải nhỏ

`PoissonRegressor` mặc định regularize khá mạnh. Với 7 feature và 7.768 trận, `alpha = 1.0` kéo hệ số của Elo từ 0.44 xuống 0.17 — mô hình gần như bỏ qua chênh lệch trình độ. Hệ quả dễ thấy nhất ở các cặp đấu lệch lớn: Spain (Elo 2214) gặp Curaçao (1613), chênh 601 điểm, Elo thuần túy cho 97% thắng nhưng mô hình chỉ cho 76%.

Hạ `alpha` xuống 0.01 thì mọi chỉ số đều tốt hơn, không phải đánh đổi:

| alpha | RPS ↓ | Đúng K/H/T |
|---|---|---|
| 1.0 | 0.1984 | 52.0% |
| 0.1 | 0.1895 | 54.8% |
| **0.01** | **0.1881** | **56.5%** |

Ridge thì không nhạy với `alpha` (0.1892 ở mọi giá trị từ 0.1 đến 100), nên giữ nguyên 10.0.

### Vì sao "đúng tỉ số" khó hơn "đúng kết quả" rất nhiều

Bài toán tỉ số là bài toán phân loại hơn 20 lớp với phân phối rất dẹt: tỉ số phổ biến nhất (1-1) chỉ chiếm khoảng 13% số trận, kế đến 0-1 và 1-0 mỗi loại chưa tới 9%. Vì vậy trần thực tế của chỉ số "đúng tỉ số" chỉ khoảng 15%, trong khi "đúng thắng/hoà/thua" có thể đạt 50–60%.

Một hệ quả quan trọng: **chọn ô xác suất lớn nhất chỉ có ích khi λ thay đổi đủ nhiều giữa các trận.** Nếu mọi trận đều có λ xấp xỉ 1,5 và 1,1 thì argmax luôn rơi vào cùng vài ô, và mô hình trở thành một hằng số trá hình. Cột "số tỉ số khác nhau" ở phần kết quả dùng để kiểm tra đúng điều này.

In [99]:
from scipy.stats import poisson

MAX_GOALS = 12      # cắt đuôi phân phối để ma trận không phình to
MIN_RATE = 0.05     # chặn dưới cho λ, tránh ma trận suy biến thành 1x1


def score_matrix(home_rate, away_rate):
    """Ma trận xác suất tỉ số, giả định số bàn hai đội độc lập theo Poisson."""
    home_rate = float(np.clip(home_rate, MIN_RATE, MAX_GOALS))
    away_rate = float(np.clip(away_rate, MIN_RATE, MAX_GOALS))
    goals = np.arange(MAX_GOALS + 1)
    probs = np.outer(poisson.pmf(goals, home_rate), poisson.pmf(goals, away_rate))
    return probs / probs.sum()


def predict_score_and_result(lambda_home, lambda_away):
    """Chọn tỉ số khả dĩ nhất và kết quả thắng/hoà/thua xác suất cao nhất."""
    predicted_home, predicted_away, predicted_result, result_probabilities = [], [], [], []
    outcomes = np.array([1, 0, -1])   # thắng nhà, hoà, thắng khách
    goals = np.arange(MAX_GOALS + 1)

    for home_rate, away_rate in zip(lambda_home, lambda_away):
        probs = score_matrix(home_rate, away_rate)
        home_index, away_index = np.unravel_index(np.argmax(probs), probs.shape)
        predicted_home.append(home_index)
        predicted_away.append(away_index)

        home_goals, away_goals = goals[:, None], goals[None, :]
        result_probs = np.array([
            probs[home_goals > away_goals].sum(),
            probs[home_goals == away_goals].sum(),
            probs[home_goals < away_goals].sum(),
        ])
        result_probs /= result_probs.sum()
        result_probabilities.append(result_probs)
        predicted_result.append(outcomes[np.argmax(result_probs)])

    return (
        np.asarray(predicted_home),
        np.asarray(predicted_away),
        np.asarray(predicted_result),
        np.asarray(result_probabilities),
    )


def evaluate(actual_home, actual_away, expected_home, expected_away):
    predicted_home, predicted_away, predicted_result, result_probabilities = (
        predict_score_and_result(expected_home, expected_away)
    )
    actual_home = actual_home.astype(int)
    actual_away = actual_away.astype(int)

    exact_match = (predicted_home == actual_home) & (predicted_away == actual_away)
    actual_result = np.sign(actual_home - actual_away)

    # RPS: so sánh xác suất tích luỹ theo thứ tự khách thắng < hoà < nhà thắng.
    actual_order = np.where(actual_result == -1, 0, np.where(actual_result == 0, 1, 2))
    predicted_cumulative = np.cumsum(result_probabilities[:, ::-1], axis=1)[:, :-1]
    actual_cumulative = actual_order[:, None] <= np.arange(2)
    rps = float(np.mean(np.mean((predicted_cumulative - actual_cumulative) ** 2, axis=1)))

    result_correct = predicted_result == actual_result
    return {
        'rps': rps,
        'result_count': int(result_correct.sum()),
        'exact_count': int(exact_match.sum()),
        'exact_accuracy': float(exact_match.mean()),
        'result_accuracy': float(result_correct.mean()),
        'distinct_scores': len(set(zip(predicted_home.tolist(), predicted_away.tolist()))),
    }


def compare_models(train_data, test_data):
    """Huấn luyện cả 4 mô hình tỉ số trên cùng một bộ feature."""
    X_train, y_home_train, y_away_train = train_data
    X_test, y_home_test, y_away_test = test_data

    results = {}
    for name, make_model in models.items():
        home_model, away_model = make_model(), make_model()
        home_model.fit(X_train, y_home_train)
        away_model.fit(X_train, y_away_train)

        expected_home = np.clip(home_model.predict(X_test), MIN_RATE, MAX_GOALS)
        expected_away = np.clip(away_model.predict(X_test), MIN_RATE, MAX_GOALS)
        results[name] = evaluate(y_home_test, y_away_test, expected_home, expected_away)
    return results


def constant_baseline(actual_home, actual_away, score):
    """Điểm của chiến lược đoán cố định, dùng làm mốc so sánh."""
    actual_home = actual_home.astype(int)
    actual_away = actual_away.astype(int)
    exact = (actual_home == score[0]) & (actual_away == score[1])
    actual_result = np.sign(actual_home - actual_away)
    predicted_result = np.sign(score[0] - score[1])
    return {
        'rps': None,
        'result_count': int((actual_result == predicted_result).sum()),
        'result_accuracy': float((actual_result == predicted_result).mean()),
        'exact_count': int(exact.sum()),
        'exact_accuracy': float(exact.mean()),
        'distinct_scores': 1,
    }


def print_table(title, rows, total_matches):
    print(title)
    print(f'{"Phương án":<34} {"RPS ↓":>7} {"Đúng K/H/T":>18} {"Đúng tỉ số":>18} {"Số tỉ số":>9}')
    for name, result in rows.items():
        rps = f'{result["rps"]:.4f}' if result.get('rps') is not None else '—'
        result_text = f'{result["result_count"]}/{total_matches} ({result["result_accuracy"]:.1%})'
        exact_text = f'{result["exact_count"]}/{total_matches} ({result["exact_accuracy"]:.1%})'
        print(f'{name:<34} {rps:>7} {result_text:>18} {exact_text:>18} '
              f'{result["distinct_scores"]:>9}')


baselines = {
    'Baseline: luôn đoán 1-1': constant_baseline(y_home_test, y_away_test, (1, 1)),
    'Baseline: luôn đoán nhà thắng': constant_baseline(y_home_test, y_away_test, (1, 0)),
}

comparison = compare_models(train_data, wc2026_data)
print_table('Test WC2026 — 7 feature', {**comparison, **baselines}, len(wc2026_data[0]))

print('\nSáu tỉ số xuất hiện nhiều nhất trong 104 trận WC 2026:')
score_counts = (
    pd.Series([f'{int(h)}-{int(a)}' for h, a in zip(y_home_test, y_away_test)])
    .value_counts(normalize=True)
    .head(6)
)
for score, share in score_counts.items():
    print(f'  {score}: {share:.1%}')


Test WC2026 — 7 feature
Phương án                            RPS ↓         Đúng K/H/T         Đúng tỉ số  Số tỉ số
Poisson Regression                  0.1602     69/104 (66.3%)     15/104 (14.4%)         7
Ridge Regression                    0.1587     69/104 (66.3%)     19/104 (18.3%)         7
Random Forest                       0.1727     66/104 (63.5%)      10/104 (9.6%)         8
Extra Trees                         0.1706     68/104 (65.4%)     13/104 (12.5%)         8
Baseline: luôn đoán 1-1                  —     24/104 (23.1%)     12/104 (11.5%)         1
Baseline: luôn đoán nhà thắng            —     50/104 (48.1%)       7/104 (6.7%)         1

Sáu tỉ số xuất hiện nhiều nhất trong 104 trận WC 2026:
  1-1: 11.5%
  2-1: 8.7%
  2-0: 7.7%
  0-1: 7.7%
  0-0: 7.7%
  1-0: 6.7%


### Phân bố kết quả và tỉ số thực tế ở tập WC 2026

Biểu đồ tóm tắt 104 trận kiểm tra: cột phải là thắng/hoà/thua, cột trái là các tỉ số xuất hiện nhiều nhất. Chỉ dùng để hiểu cấu trúc tập kiểm tra, không phải feature đưa vào huấn luyện.

![Phân bố kết quả và tỉ số WC 2026](../figures/train_02_wc2026_scores.png)

## 4. Đọc các chỉ số

- **RPS (Ranked Probability Score)**: chỉ số chính, đánh giá chất lượng xác suất thắng/hoà/thua theo đúng thứ tự khách thắng < hoà < nhà thắng. Càng thấp càng tốt, 0 là hoàn hảo. Ưu điểm so với accuracy: phạt nhẹ khi đoán "gần đúng" và phạt nặng khi đoán ngược hoàn toàn.
- **Đúng K/H/T**: chỉ số chính thứ hai, chọn kết quả có tổng xác suất cao nhất rồi tính tỉ lệ đúng.
- **Đúng tỉ số**: chỉ số tham khảo. Chọn ô có xác suất lớn nhất trong ma trận Poisson.
- **Số tỉ số**: số tỉ số khác nhau mà mô hình dự đoán cho cả 104 trận. Đây là chỉ số chẩn đoán, không phải chỉ số chất lượng. Nếu một mô hình chỉ đưa ra 4–5 tỉ số khác nhau thì argmax của nó đang bị "kẹt" ở vài ô, và độ chính xác tỉ số sẽ quanh mức của việc đoán hằng số.
- **Baseline**: mốc so sánh bắt buộc. "Luôn đoán 1-1" thường đạt khoảng 11–13% đúng tỉ số.

Đúng tỉ số chính xác là bài toán 20+ lớp với phân phối rất dẹt, trần thực tế chỉ khoảng 15%. Vì vậy nên ưu tiên RPS và đúng K/H/T, còn đúng tỉ số chỉ dùng để tham khảo.

## 5. Mô hình 1X2 trực tiếp: Multinomial Logit

Bốn mô hình trên đều đi vòng: dự đoán **số bàn** rồi cộng xác suất để ra thắng/hoà/thua. Nhưng mục tiêu cuối cùng chỉ là 1X2, nên có thể học thẳng xác suất của 3 kết quả bằng **Multinomial Logistic Regression**.

Mô hình dùng 2 biến:

- `elo_diff` = Elo đội nhà − Elo đội khách
- `home_adv` = 1 nếu đội nhà thật sự có lợi thế sân nhà, 0 nếu sân trung lập

Khác biệt về bản chất: mô hình tỉ số phải mô tả **phân phối số bàn** (một bài toán khó, nhiều nhiễu), rồi mới suy ra 1X2. Multinomial Logit bỏ qua bước trung gian đó và tối ưu trực tiếp đại lượng ta cần đo.

Đánh đổi: mô hình này **không cho biết tỉ số**, nên không dùng được để mô phỏng giải đấu (cần số bàn để tính hiệu số, xếp hạng, hiệp phụ, luân lưu).


### Kết quả 1X2 theo chênh lệch Elo

Biểu đồ dùng các trận sân trung lập trong dữ liệu lịch sử để minh hoạ tỉ lệ thắng/hoà/thua theo chênh lệch Elo. Đây là động lực cho mô hình 1X2 trực tiếp ở ô kế tiếp; các tỉ lệ quan sát không phải xác suất đã hiệu chỉnh.

![Tỉ lệ 1X2 theo chênh lệch Elo](../figures/train_03_elo_outcome.png)

In [100]:
import statsmodels.api as sm


def make_1x2(rows):
    """Ma trận thiết kế cho Multinomial Logit. Nhãn: 0 khách thắng, 1 hoà, 2 nhà thắng."""
    elo_diff = np.asarray([float(r['home_elo_pre']) - float(r['away_elo_pre']) for r in rows])
    home_adv = np.asarray([0.0 if parse_bool(r['neutral']) else 1.0 for r in rows])
    X = sm.add_constant(np.column_stack([elo_diff, home_adv]), has_constant='add')
    y = np.asarray([
        2 if float(r['home_score']) > float(r['away_score'])
        else 1 if float(r['home_score']) == float(r['away_score']) else 0
        for r in rows
    ])
    return X, y


def rps_1x2(y, probs):
    """probs có 3 cột theo thứ tự [khách thắng, hoà, nhà thắng]."""
    cumulative = np.cumsum(probs, axis=1)[:, :-1]
    actual = y[:, None] <= np.arange(2)
    return float(np.mean(np.sum((cumulative - actual) ** 2, axis=1) / 2))


def fit_mnlogit(train_rows, test_rows):
    X_train, y_train = make_1x2(train_rows)
    X_test, y_test = make_1x2(test_rows)
    fitted = sm.MNLogit(y_train, X_train).fit(disp=0, maxiter=500)
    probs = np.asarray(fitted.predict(X_test))
    return fitted, probs, y_test


model_mnlogit, probs_mnlogit, y_wc = fit_mnlogit(history, world_cup)

print(f'{"Mô hình 1X2":<40} {"RPS ↓":>7} {"Đúng K/H/T":>18}')
correct = int((probs_mnlogit.argmax(axis=1) == y_wc).sum())
accuracy = f'{correct}/{len(y_wc)} ({correct / len(y_wc):.1%})'
print(f'{"MNLogit: elo + sân nhà":<40} {rps_1x2(y_wc, probs_mnlogit):>7.4f} {accuracy:>18}')

print('\nHệ số mô hình (tham chiếu: đội khách thắng):')
coefficient_names = ['const', 'elo_diff', 'home_adv']
for position, label in [(0, 'Hòa'), (1, 'Đội nhà thắng')]:
    # X đưa vào dưới dạng numpy nên params là ndarray, không phải DataFrame.
    values = model_mnlogit.params[:, position]
    row = '  '.join(f'{name} {value:+.4f}'
                    for name, value in zip(coefficient_names, values))
    print(f'  {label:<14} {row}')

print('\nÝ nghĩa: hệ số dương của elo_diff ở nhánh "đội nhà thắng" nghĩa là '
      'đội nhà mạnh hơn theo Elo thì khả năng thắng tăng.')


Mô hình 1X2                                RPS ↓         Đúng K/H/T
MNLogit: elo + sân nhà                    0.1566     69/104 (66.3%)

Hệ số mô hình (tham chiếu: đội khách thắng):
  Hòa            const -0.1377  elo_diff +0.0035  home_adv +0.2940
  Đội nhà thắng  const +0.0848  elo_diff +0.0072  home_adv +0.6023

Ý nghĩa: hệ số dương của elo_diff ở nhánh "đội nhà thắng" nghĩa là đội nhà mạnh hơn theo Elo thì khả năng thắng tăng.


### So sánh RPS trên WC 2026

Biểu đồ so sánh RPS của các mô hình trên cùng 104 trận; cột thấp hơn tốt hơn. Chênh lệch nhỏ cần được đọc thận trọng vì tập kiểm tra còn ít trận và kết quả WC 2026 đã được dùng trong thử nghiệm.

![So sánh RPS các mô hình trên WC 2026](../figures/train_04_rps_compare.png)

## 6. Backtest trên 12 giải

RPS trên riêng WC 2026 không đủ tin cậy: sai số chuẩn khoảng ±0.012, trong khi chênh lệch giữa các mô hình chỉ 0.002–0.005. Cách khắc phục là **walk-forward**: với mỗi giải, huấn luyện trên toàn bộ dữ liệu **trước ngày khai mạc** rồi kiểm tra đúng các trận của giải đó. Không giải nào dùng dữ liệu của chính nó.

Bảng dưới so 3 phương án:

| Ký hiệu | Mô hình |
|---|---|
| `MNLogit` | 1X2 trực tiếp, Elo + sân nhà |
| `Poisson` | mô hình tỉ số, 7 feature |
| `Ridge` | mô hình tỉ số, 7 feature |

Lưu ý: RPS khác nhau rất nhiều giữa các giải (AFCON và WC khó hơn Copa América), nên chỉ nên so **trong cùng một dòng**, không so giữa các dòng.


In [101]:
TOURNAMENTS = {
    'Copa America 2019': ('2019-06-14', '2019-07-07', 'Copa América'),
    'Asian Cup 2019': ('2019-01-05', '2019-02-01', 'AFC Asian Cup'),
    'AFCON 2019': ('2019-06-21', '2019-07-19', 'African Cup of Nations'),
    'Euro 2021': ('2021-06-11', '2021-07-11', 'UEFA Euro'),
    'Copa America 2021': ('2021-06-13', '2021-07-10', 'Copa América'),
    'AFCON 2021': ('2022-01-09', '2022-02-06', 'African Cup of Nations'),
    'WC 2022': ('2022-11-20', '2022-12-18', 'FIFA World Cup'),
    'Asian Cup 2023': ('2024-01-12', '2024-02-10', 'AFC Asian Cup'),
    'AFCON 2023': ('2024-01-13', '2024-02-11', 'African Cup of Nations'),
    'Copa America 2024': ('2024-06-20', '2024-07-14', 'Copa América'),
    'Euro 2024': ('2024-06-14', '2024-07-14', 'UEFA Euro'),
    'WC 2026': ('2026-06-11', '2026-07-19', 'FIFA World Cup'),
}

# Mỗi giải chỉ được huấn luyện trên các trận TRƯỚC ngày khai mạc của giải đó.
all_rows = history + world_cup
walk_rows = []

for name, (start, end, tournament) in TOURNAMENTS.items():
    test_rows = [r for r in all_rows
                 if start <= r['date'] <= end and r['tournament'] == tournament]
    train_rows = [r for r in all_rows if r['date'] < start]
    if len(test_rows) < 15:
        continue

    _, p_mnlogit, y_test = fit_mnlogit(train_rows, test_rows)
    score_models = compare_models(make_data(train_rows), make_data(test_rows))

    walk_rows.append({
        'Giải': name,
        'n': len(test_rows),
        'MNLogit': rps_1x2(y_test, p_mnlogit),
        'Poisson': score_models['Poisson Regression']['rps'],
        'Ridge': score_models['Ridge Regression']['rps'],
    })

walk = pd.DataFrame(walk_rows)
print(walk.round(4).to_string(index=False))

weights = walk['n']
print()
print(f'Trung bình có trọng số theo số trận ({int(weights.sum())} trận / {len(walk)} giải):')
for column in ['MNLogit', 'Poisson', 'Ridge']:
    print(f'  {column:<10} {np.average(walk[column], weights=weights):.4f}')


             Giải   n  MNLogit  Poisson  Ridge
Copa America 2019  26   0.1543   0.1631 0.1621
   Asian Cup 2019  51   0.1668   0.1727 0.1681
       AFCON 2019  52   0.1779   0.1885 0.1896
        Euro 2021  51   0.1782   0.1786 0.1752
Copa America 2021  28   0.1567   0.1562 0.1564
       AFCON 2021  52   0.2098   0.2017 0.2096
          WC 2022  64   0.2272   0.2260 0.2323
   Asian Cup 2023  51   0.1677   0.1662 0.1682
       AFCON 2023  52   0.2228   0.2141 0.2245
Copa America 2024  32   0.1527   0.1535 0.1499
        Euro 2024  51   0.1956   0.1990 0.2042
          WC 2026 104   0.1566   0.1602 0.1587

Trung bình có trọng số theo số trận (614 trận / 12 giải):
  MNLogit    0.1824
  Poisson    0.1834
  Ridge      0.1852


## 7. Kết luận và giới hạn

**Mô hình đã gần trần của bộ feature hiện tại.** Bảy feature (Elo, sân nhà, phong độ 5 trận) cho RPS khoảng 0.155–0.160 trên WC 2026 và 0.18 trên 12 giải backtest. Chênh lệch giữa bốn mô hình tỉ số và Multinomial Logit đều nhỏ hơn sai số chuẩn của tập test, nên đổi mô hình không còn tác dụng.

**Đã thử và loại:** thêm ngày nghỉ, phong độ dài hơn, spline Elo, dummy đội, trọng số thời gian, hiệu chỉnh nhiệt độ, Dixon–Coles, Negative Binomial, và giá trị đội hình theo Transfermarkt. Giá trị đội hình là cái duy nhất cải thiện được — nhưng chỉ khoảng 0.002–0.003 RPS, chưa bằng một phần tư sai số chuẩn của tập test, nên đã bỏ khỏi pipeline để giữ mô hình gọn.


## 8. Mô phỏng toàn giải World Cup 2026

Chỉ in **tỉ số**, không ghi file. Code chia làm 2 phần, chạy lần lượt:

- **8.1 Vòng bảng** — 72 trận, xếp hạng bảng, gán 8 đội hạng ba vào bracket.
- **8.2 Vòng loại trực tiếp** — R32 → Chung kết + trận hạng ba, in từ final ngược về.

Cell đầu tiên (8.0) nạp dữ liệu, huấn luyện Poisson và định nghĩa các hàm dùng chung.

### Nguyên tắc: không chỉnh mô hình theo kết quả 2026

Bản này **cố tình để các hệ số ở mức trung tính**: `LAMBDA_SCALE_HOME = LAMBDA_SCALE_AWAY = 1.0`, `OVERDISPERSION_SIGMA = 0.0`, seed mặc định không được chọn để tái tạo một kết quả cụ thể. λ lấy thẳng từ PoissonRegression(`alpha=0.01`) huấn luyện trên 7.768 trận lịch sử; Elo, phong độ, bracket, hiệp phụ λ×0.35 và luân lưu theo Elo đều giữ nguyên logic gốc.

**Vì sao không hiệu chỉnh cho "giống 2026":**

| Thử nghiệm | Bàn/trận | RPS trên 2026 |
|---|---|---|
| Mô hình gốc (bản hiện tại) | 2.67 | **0.1602** |
| Scale ước từ 11 giải trước 2026 | 2.51 | 0.1618 (tệ hơn) |
| Scale ước từ chính 104 trận 2026 | 2.96 | 0.1581 (nhưng là fit trên tập test) |

Tỉ lệ `bàn thực / λ` dao động 0.78–1.11 giữa các giải (AFCON 2021 = 0.78, Copa 2024 = 0.87, Euro 2024 = 0.92, WC 2022 = 1.11, WC 2026 = 1.11). WC 2026 là giải ghi bàn nhiều bất thường, không phải lỗi hệ thống của mô hình. Áp hệ số lấy từ các giải trước sang thì RPS tệ hơn, nên **không có cơ sở hợp lệ để scale λ cho riêng 2026**. Chọn seed để ra đúng Spain 1-0 Argentina cũng chỉ là chọn một lần quay đẹp, không phải chất lượng dự báo.

Vì vậy output của bản này có thể không giống 2026 — đó là mong đợi. Muốn cải thiện thật cho 2030 cần dữ liệu mới (xG, tỉ lệ cược, chấn thương), không phải tuning trên 104 trận đã biết.

Giả định giải đấu: xếp hạng theo điểm → hiệu số → bàn ghi → Elo; 8 đội hạng ba tốt nhất; hiệp phụ λ×0.35; luân lưu theo logistic Elo. `SIMULATION_RUNS = 1000` dùng cho bảng **PHÂN PHỐI XÁC SUẤT** ở cuối cell 8.2 (cộng dồn đủ 1.000 lần); phần tỉ số là **một giải liên tục duy nhất**: cell 8.1 đá vòng bảng, cell 8.2 nối tiếp state và RNG để đá knockout. Tăng `SIMULATION_RUNS` để bảng xác suất ổn định hơn. Seed trung tính là `2030`; hiện đang dùng seed demo `10929` theo yêu cầu (xem mục **8.3**), không dùng để đánh giá mô hình. Đổi `RANDOM_SEED = 2030` để quay lại cấu hình trung tính.


In [102]:
from collections import defaultdict, deque
from itertools import combinations

import copy
import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
THIRD_PLACE_CSV = ROOT / 'dataset' / 'mo_phong' / 'third_place_combinations_wc2026.csv'
third_place_table = pd.read_csv(THIRD_PLACE_CSV, dtype=str).fillna('')

# Bản tối ưu: chỉ in tỉ số, không ghi file. Cộng dồn 1.000 lần cho bảng xác suất.
SIMULATION_RUNS = 1000
# Seed demo ĐƯỢC CHỌN theo kết quả 2026: cherry-picking, KHÔNG phải cải thiện mô hình.
DEMO_SEED_2026 = 10929
# Đang dùng seed demo theo yêu cầu. Đổi về 2030 để quay lại cấu hình trung tính.
RANDOM_SEED = DEMO_SEED_2026
# λ luôn trung tính: không scale, không thêm nhiễu. Chỉ seed đang là demo (xem mục 8.3).
LAMBDA_SCALE_HOME = 1.0
LAMBDA_SCALE_AWAY = 1.0
OVERDISPERSION_SIGMA = 0.0
rng = np.random.default_rng(RANDOM_SEED)

GROUP_CSV = ROOT / 'dataset' / 'mo_phong' / 'group_stages.csv'
KNOCKOUT_CSV = ROOT / 'dataset' / 'mo_phong' / 'fixtures_knockout_wc2026.csv'
group_table = pd.read_csv(GROUP_CSV, sep=';')
knockout_table = pd.read_csv(KNOCKOUT_CSV)
wc_features = pd.read_csv(ROOT / 'loc' / 'wc_2026_da_loc.csv')

team_states = {}
for team in sorted(set(group_table['nation'])):
    matches = wc_features[
        (wc_features['home_team'] == team) | (wc_features['away_team'] == team)
    ]
    if matches.empty:
        raise ValueError(f'Thiếu đặc trưng Elo/phong độ cho đội {team}')
    first = matches.iloc[0]
    if first['home_team'] == team:
        elo = float(first['home_elo_pre'])
        gf = float(first['home_form_goals_scored_last5'])
        ga = float(first['home_form_goals_conceded_last5'])
    else:
        elo = float(first['away_elo_pre'])
        gf = float(first['away_form_goals_scored_last5'])
        ga = float(first['away_form_goals_conceded_last5'])
    team_states[team] = {
        'elo': elo,
        'gf': deque([gf] * 5, maxlen=5),
        'ga': deque([ga] * 5, maxlen=5),
    }

# --- Tối ưu 1: lấy hệ số tuyến tính của pipeline thay vì gọi predict() từng trận ---
# StandardScaler -> PoissonRegressor tương đương exp(X @ w + b).
X_history, y_home_history, y_away_history = make_data(history)


def linear_coefficients(y_target):
    model = make_pipeline(
        StandardScaler(), PoissonRegressor(alpha=0.01, max_iter=3000)
    )
    model.fit(X_history, y_target)
    scaler = model.named_steps['standardscaler']
    reg = model.named_steps['poissonregressor']
    scale = np.asarray(scaler.scale_, dtype=float)
    coef = np.asarray(reg.coef_, dtype=float).ravel()
    weights = coef / scale
    bias = float(reg.intercept_ - np.dot(coef, np.asarray(scaler.mean_) / scale))
    return weights, bias


W_HOME, B_HOME = linear_coefficients(y_home_history)
W_AWAY, B_AWAY = linear_coefficients(y_away_history)

groups = {
    group: group_table.loc[group_table['group'] == group, 'nation'].tolist()
    for group in sorted(group_table['group'].unique())
}
group_pairs = {
    group: list(combinations(teams, 2)) for group, teams in groups.items()
}
team_to_group = {
    team: group for group, teams in groups.items() for team in teams
}
knockout_fixtures = list(knockout_table.itertuples(index=False))


def predict_lambda(home_team, away_team, states):
    home, away = states[home_team], states[away_team]
    x = np.array([
        home['elo'], away['elo'], 1.0,
        sum(home['gf']) / 5, sum(home['ga']) / 5,
        sum(away['gf']) / 5, sum(away['ga']) / 5,
    ])
    home_lambda = float(np.clip(np.exp(x @ W_HOME + B_HOME) * LAMBDA_SCALE_HOME, 0.05, 12.0))
    away_lambda = float(np.clip(np.exp(x @ W_AWAY + B_AWAY) * LAMBDA_SCALE_AWAY, 0.05, 12.0))
    return home_lambda, away_lambda


def expected_home(home_team, away_team, states):
    diff = states[home_team]['elo'] - states[away_team]['elo']
    return 1.0 / (1.0 + 10 ** (-diff / 400))


def update_state(home_team, away_team, home_goals, away_goals, states):
    home, away = states[home_team], states[away_team]
    exp_home = expected_home(home_team, away_team, states)
    if home_goals > away_goals:
        actual_home = 1.0
    elif home_goals < away_goals:
        actual_home = 0.0
    else:
        actual_home = 0.5
    goal_diff = abs(home_goals - away_goals)
    if goal_diff <= 1:
        multiplier = 1.0
    elif goal_diff == 2:
        multiplier = 1.5
    else:
        multiplier = (11 + goal_diff) / 8
    delta = 60.0 * multiplier * (actual_home - exp_home)
    home['elo'] += delta
    away['elo'] -= delta
    home['gf'].append(float(home_goals))
    home['ga'].append(float(away_goals))
    away['gf'].append(float(away_goals))
    away['ga'].append(float(home_goals))


def draw_match(home_team, away_team, states, match_id, round_name):
    home_lambda, away_lambda = predict_lambda(home_team, away_team, states)
    # Nhiễu dùng CHUNG cho hai đội: tổng bàn phân tán hơn nhưng vẫn
    # giữ tương quan hợp lý giữa hai λ (khác với nhiễu riêng từng đội).
    if OVERDISPERSION_SIGMA:
        shock = np.exp(rng.normal(0.0, OVERDISPERSION_SIGMA))
        home_lambda *= shock
        away_lambda *= shock

    # Tối ưu 2: RNG của NumPy, không dùng scipy.stats.poisson.rvs
    home_goals = int(rng.poisson(home_lambda))
    away_goals = int(rng.poisson(away_lambda))
    extra_time = False
    penalties = False
    if round_name in ('R32', 'R16', 'QF', 'SF', '3rd', 'Final'):
        if home_goals == away_goals:
            extra_time = True
            home_goals += int(rng.poisson(home_lambda * 0.35))
            away_goals += int(rng.poisson(away_lambda * 0.35))
        if home_goals == away_goals:
            penalties = True
            winner = (
                home_team
                if rng.random() < expected_home(home_team, away_team, states)
                else away_team
            )
        else:
            winner = home_team if home_goals > away_goals else away_team
    else:
        winner = None
    update_state(home_team, away_team, home_goals, away_goals, states)
    record = (
        match_id, round_name, home_team, away_team,
        home_goals, away_goals, extra_time, penalties, winner,
    )
    return record


def fresh_states():
    """Tạo bản sao Elo/phong độ mới cho một lần chạy giải."""
    return {
        team: {
            'elo': float(state['elo']),
            'gf': deque(state['gf'], maxlen=5),
            'ga': deque(state['ga'], maxlen=5),
        }
        for team, state in team_states.items()
    }


def simulate_group_stage(states):
    """12 bảng x 4 đội, vong tron 6 tran/bang. Tra ve records, bang xep hang va slot."""
    records = []
    standings = {}

    for group, teams in groups.items():
        stats = {team: [0, 0, 0] for team in teams}  # diem, ban ghi, ban thung
        for number, (home_team, away_team) in enumerate(
            group_pairs[group], start=1
        ):
            record = draw_match(
                home_team, away_team, states,
                f'{group}-M{number}', 'Vòng bảng',
            )
            records.append(record)
            _, _, _, _, home_goals, away_goals, _, _, _ = record
            home_stats, away_stats = stats[home_team], stats[away_team]
            home_stats[1] += home_goals
            home_stats[2] += away_goals
            away_stats[1] += away_goals
            away_stats[2] += home_goals
            if home_goals > away_goals:
                home_stats[0] += 3
            elif home_goals < away_goals:
                away_stats[0] += 3
            else:
                home_stats[0] += 1
                away_stats[0] += 1

        overall_key = lambda team: (
            stats[team][0],
            stats[team][1] - stats[team][2],
            stats[team][1],
        )
        ranked = sorted(teams, key=overall_key, reverse=True)

        # Nếu vẫn bằng sau điểm, hiệu số và bàn thắng, xét thành tích đối đầu.
        i = 0
        while i < len(ranked):
            j = i + 1
            while j < len(ranked) and overall_key(ranked[j]) == overall_key(ranked[i]):
                j += 1
            tied = ranked[i:j]
            if len(tied) > 1:
                tied_set = set(tied)
                h2h = {}
                for team in tied:
                    points = gf = ga = 0
                    for match in records:
                        if match[1] != 'Vòng bảng':
                            continue
                        _, _, home, away, hg, ag, _, _, _ = match
                        if home not in tied_set or away not in tied_set:
                            continue
                        if home == team:
                            gf += hg
                            ga += ag
                            points += 3 if hg > ag else 1 if hg == ag else 0
                        elif away == team:
                            gf += ag
                            ga += hg
                            points += 3 if ag > hg else 1 if ag == hg else 0
                    h2h[team] = (points, gf - ga, gf)
                tied.sort(key=lambda team: h2h[team], reverse=True)
                ranked[i:j] = tied
            i = j
        standings[group] = (ranked, stats)

# Xếp hạng ba theo điểm, hiệu số và bàn thắng (không dùng Elo).
    third_places = []
    for group, (ranked, stats) in standings.items():
        team = ranked[2]
        points, gf, ga = stats[team]
        third_places.append((team, points, gf - ga, gf))
    third_places.sort(
        key=lambda row: (row[1], row[2], row[3]),
        reverse=True,
    )
    third_ranked = [row[0] for row in third_places]

    # Chọn đúng tổ hợp từ bảng 495 kịch bản của FIFA.
    qualified_third_groups = ''.join(sorted(team_to_group[team] for team in third_ranked[:8]))
    combination = third_place_table[
        third_place_table['qualified_groups'] == qualified_third_groups
    ]
    if combination.empty:
        raise RuntimeError(f'Không tìm thấy tổ hợp đội hạng ba FIFA: {qualified_third_groups}')
    option = combination.iloc[0]
    slot_to_team = {}
    for first_slot in ('1A', '1B', '1D', '1E', '1G', '1I', '1K', '1L'):
        third_group = option[first_slot]
        team = next(
            team for team in third_ranked[:8]
            if team_to_group[team] == third_group
        )
        slot_to_team[f'T{first_slot}'] = team
    return records, standings, third_ranked, slot_to_team


def simulate_knockout(states, standings, slot_to_team):
    """R32 -> R16 -> QF -> SF -> Final, them tran hang ba."""
    winners = {}
    losers = {}

    def resolve_slot(slot):
        if slot.startswith('T'):
            return slot_to_team[slot]
        if slot.startswith('W'):
            return winners[slot]
        if slot.startswith('RU'):
            return losers[slot]
        if len(slot) == 2 and slot[0].isdigit():
            return standings[slot[1]][0][int(slot[0]) - 1]
        if slot.startswith('3') and len(slot) > 1:
            return slot_to_team[slot]
        raise RuntimeError(f'Slot không hợp lệ: {slot}')

    records = []
    for fixture in knockout_fixtures:
        if fixture.round not in ('R32', 'R16', 'QF', 'SF', '3rd', 'Final'):
            continue
        home_team = resolve_slot(fixture.home_slot)
        away_team = resolve_slot(fixture.away_slot)
        record = draw_match(
            home_team, away_team, states,
            fixture.match_id, fixture.round,
        )
        records.append(record)
        winner = record[-1]
        winners[f'W{fixture.match_id[1:]}'] = winner
        losers[f'RU{fixture.match_id[1:]}'] = (
            away_team if winner == home_team else home_team
        )
    return records


ROUND_ORDER = {
    'Final': 0, '3rd': 1, 'SF': 2, 'QF': 3,
    'R16': 4, 'R32': 5,
}
ROUND_NAME = {
    'Final': 'CHUNG KẾT',
    '3rd': 'TRẬN HẠNG BA',
    'SF': 'BÁN KẾT',
    'QF': 'TỨ KẾT',
    'R16': 'VÒNG 1/16',
    'R32': 'VÒNG 1/32',
}
SEPARATOR = '=' * 68


def print_score_line(match):
    match_id, _, home, away, home_goals, away_goals, extra, penalties, _ = match
    score = f'{home_goals}-{away_goals}'
    note = ''
    if penalties:
        note = '  [hiệp phụ + luân lưu]'
    elif extra:
        note = '  [sau hiệp phụ]'
    print(f'  {match_id:<6} {home:>25} {score:^7} {away:<25}{note}')


### 8.1 Vòng bảng

Chạy cell này **trước** cell 8.2. Mỗi bảng 4 đội, đá vòng tròn 6 trận → tổng 72 trận. Xếp hạng: điểm → hiệu số bàn → bàn ghi → thành tích đối đầu.

Cell cũng lưu `states`, `standings` và `slot_to_team` theo tổ hợp FIFA 495 kịch bản để cell 8.2 dùng tiếp. In ra **PHẦN 1 — VÒNG BẢNG**, các bảng từ A đến L.


In [103]:
# MỘT giải HIỂN THỊ duy nhất: vòng bảng.
# Cell 8.2 sẽ nối tiếp chính state và RNG này để đá vòng loại.
rng = np.random.default_rng(RANDOM_SEED)
states = fresh_states()
last_group_records, last_standings, _, last_slot_to_team = simulate_group_stage(states)
last_states = states
display_rng_state = copy.deepcopy(rng.bit_generator.state)

print(SEPARATOR)
print(f'WC 2026 — {SIMULATION_RUNS:,} LẦN MÔ PHỎNG (seed {RANDOM_SEED})')
print(SEPARATOR)
print()
print('PHẦN 1 — VÒNG BẢNG (cùng giải với phần 8.2)')

print(SEPARATOR)
for group in sorted(groups):
    print(f'--- Bảng {group} ---')
    for match in last_group_records:
        if team_to_group[match[2]] == group:
            print_score_line(match)
    print()


WC 2026 — 1,000 LẦN MÔ PHỎNG (seed 10929)

PHẦN 1 — VÒNG BẢNG (cùng giải với phần 8.2)
--- Bảng A ---
  A-M1                      Mexico   3-1   South Africa             
  A-M2                      Mexico   2-0   South Korea              
  A-M3                      Mexico   1-1   Czech Republic           
  A-M4                South Africa   1-0   South Korea              
  A-M5                South Africa   2-2   Czech Republic           
  A-M6                 South Korea   2-1   Czech Republic           

--- Bảng B ---
  B-M1                      Canada   2-0   Bosnia and Herzegovina   
  B-M2                      Canada   3-0   Qatar                    
  B-M3                      Canada   2-2   Switzerland              
  B-M4      Bosnia and Herzegovina   4-0   Qatar                    
  B-M5      Bosnia and Herzegovina   0-3   Switzerland              
  B-M6                       Qatar   2-4   Switzerland              

--- Bảng C ---
  C-M1                      Brazil   1

### 8.2 Vòng loại trực tiếp

Chạy sau cell 8.1. Bracket R32 → R16 → Tứ kết → Bán kết → Chung kết, cộng trận hạng ba. Tỉ số in theo thứ tự **từ chung kết ngược về vòng 1/32**.

Trận hoà sau 90 phút: thêm 2 nửa giờ bù với λ×0.35. Vẫn hoà: luân lưu, xác suất thắng lấy từ logistic Elo. Tên hai đội được căn đối xứng qua tỉ số.


In [104]:
# Bảng xác suất: SIMULATION_RUNS giải bắt đầu từ RANDOM_SEED.
# Tỉ số in ra là phần nối tiếp của giải hiển thị ở cell 8.1 (cùng seed, cùng RNG).
STAGE_KEYS = ['R32', 'R16', 'QF', 'SF', 'Final', 'Champion']
stage_counts = {team: defaultdict(int) for team in team_states}
knockout_records_last = []

rng = np.random.default_rng(RANDOM_SEED)
for _ in range(SIMULATION_RUNS):
    states = fresh_states()
    group_records, standings, third_ranked, slot_to_team = simulate_group_stage(states)
    qualified = [
        standings[group][0][rank - 1]
        for group in standings for rank in (1, 2)
    ] + list(slot_to_team.values())
    for team in qualified:
        stage_counts[team]['R32'] += 1

    knockout_records = simulate_knockout(states, standings, slot_to_team)
    winner_by_match = {
        match[0]: (match[2] if match[4] > match[5] else match[3])
        for match in knockout_records
        if match[4] != match[5]
    }
    # Trận hoà được quyết định bằng hiệp phụ/luân lưu: winner nằm ở record[-1].
    winner_by_match.update({
        match[0]: match[-1] for match in knockout_records if match[4] == match[5]
    })
    for match in knockout_records:
        if match[1] == 'R32':
            stage_counts[winner_by_match[match[0]]]['R16'] += 1
        elif match[1] == 'R16':
            stage_counts[winner_by_match[match[0]]]['QF'] += 1
        elif match[1] == 'QF':
            stage_counts[winner_by_match[match[0]]]['SF'] += 1
        elif match[1] == 'SF':
            stage_counts[winner_by_match[match[0]]]['Final'] += 1
    final_match = [match for match in knockout_records if match[1] == 'Final'][0]
    stage_counts[final_match[-1]]['Champion'] += 1

# Tiếp nối GIẢI HIỂN THỊ từ 8.1: dùng lại bảng xếp hạng và đúng trạng thái
# RNG ngay sau 72 trận bảng. Không chạy lại vòng bảng, không reset seed.
rng.bit_generator.state = copy.deepcopy(display_rng_state)
knockout_records_last = simulate_knockout(last_states, last_standings, last_slot_to_team)

print('PHẦN 2 — VÒNG LOẠI TRỰC TIẾP')
print(SEPARATOR)
last_round_name = None
for match in sorted(knockout_records_last, key=lambda m: (ROUND_ORDER[m[1]], m[0])):
    round_name = match[1]
    if round_name != last_round_name:
        print()
        print(ROUND_NAME[round_name])
        print('-' * 68)
        last_round_name = round_name
    print_score_line(match)

print()
print(f'PHÂN PHỐI XÁC SUẤT — {SIMULATION_RUNS:,} LẦN')
print(SEPARATOR)
HEADERS = ['R32', 'R16', 'Tứ kết', 'Bán kết', 'Chung kết', 'Vô địch']
print(f'{"Đội":<25}' + ''.join(f'{h:>11}' for h in HEADERS))
for team in sorted(team_states, key=lambda t: (-stage_counts[t]['Champion'], t))[:15]:
    values = [stage_counts[team][key] / SIMULATION_RUNS for key in STAGE_KEYS]
    print(f'{team:<25}' + ''.join(f'{value:>10.1%} ' for value in values))

total_champion = sum(stage_counts[t]['Champion'] for t in team_states) / SIMULATION_RUNS
print(f'\nTổng xác suất vô địch: {total_champion:.2%}')


PHẦN 2 — VÒNG LOẠI TRỰC TIẾP

CHUNG KẾT
--------------------------------------------------------------------
  M104                       Spain   1-0   Argentina                

TRẬN HẠNG BA
--------------------------------------------------------------------
  M103                      France   2-1   Belgium                  

BÁN KẾT
--------------------------------------------------------------------
  M101                      France   0-0   Spain                      [hiệp phụ + luân lưu]
  M102                   Argentina   0-0   Belgium                    [hiệp phụ + luân lưu]

TỨ KẾT
--------------------------------------------------------------------
  M100                     England   0-1   Belgium                  
  M97                        Japan   0-1   France                   
  M98                  Switzerland   0-1   Spain                    
  M99                    Argentina   2-1   Brazil                     [sau hiệp phụ]

VÒNG 1/16
----------------------------

In [105]:
final_match = next(match for match in knockout_records_last if match[1] == 'Final')
champion = final_match[-1]
runner_up = final_match[3] if champion == final_match[2] else final_match[2]
third_match = next(match for match in knockout_records_last if match[1] == '3rd')
third_place = third_match[-1]
fourth_place = third_match[3] if third_place == third_match[2] else third_match[2]
finish_order = [champion, runner_up, third_place, fourth_place]

for round_name in ('QF', 'R16', 'R32'):
    eliminated = []
    for match in knockout_records_last:
        if match[1] != round_name:
            continue
        winner = match[-1]
        loser = match[3] if winner == match[2] else match[2]
        loser_goals = match[5] if loser == match[3] else match[4]
        goal_margin = abs(match[4] - match[5])
        eliminated.append((loser, goal_margin, loser_goals))
    eliminated.sort(key=lambda item: (item[1], -item[2], item[0]))
    finish_order.extend(team for team, _, _ in eliminated)

print()
print(f'ĐỘI VÔ ĐỊCH: {champion}')
print('TOP 10 ĐỘI XẾP HẠNG)')
print(SEPARATOR)
for rank, team in enumerate(finish_order[:10], start=1):
    print(f'{rank:>2}. {team}')


ĐỘI VÔ ĐỊCH: Spain
TOP 10 ĐỘI XẾP HẠNG)
 1. Spain
 2. Argentina
 3. France
 4. Belgium
 5. Brazil
 6. England
 7. Japan
 8. Switzerland
 9. Canada
10. Portugal
